# Lab 1: Manufacturing Data Setup

## What this lab does:

Creates a realistic manufacturing dataset with 4 tables in Unity Catalog:

1. **machines** - 50 production machines with installation info
2. **sensor_readings** - 10K IoT sensor readings (temperature, vibration, load)
3. **production_orders** - 200 manufacturing orders
4. **maintenance_tickets** - 120 maintenance tickets

## Data Structure:

```
catalog_workshop
  └─ schema_{your_name}
       ├─ machines (50 rows)
       ├─ sensor_readings (10,000 rows)
       ├─ production_orders (200 rows)
       └─ maintenance_tickets (120 rows)
```

## Usage:

This dataset will be used in **Lab2** as the source for Lakebase Postgres sync and Change Data Feed.

Run all cells in order to create the tables.

In [0]:
import re

# Define catalog and schema names
CATALOG = "catalog_workshop"  # Shared catalog for workshop
user = spark.sql("SELECT current_user()").first()[0]

# Create a unique schema name from your email (removes dots, special chars)
# Example: user.name@company.com -> username -> schema_username
slug = re.sub(r"[^a-z0-9]", "", user.split("@")[0].lower())
schema = f"schema_{slug}"

# Create the Unity Catalog structure
# IF NOT EXISTS prevents errors if already created
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")

print(f"✅ Your target location: {CATALOG}.{schema}")
print(f"   All tables will be created here as Delta tables")

## 🏭 About the Manufacturing Dataset

We're creating a **realistic IoT manufacturing dataset** with 4 interconnected tables:

### Why these tables?

1. **machines** - The physical assets in your factory
   * Think of this as your equipment inventory
   * Each machine has an ID, model, location, and installation date

2. **sensor_readings** - Real-time IoT data from sensors
   * Temperature, vibration, and load measurements
   * 10,000 readings spanning 30 days
   * This is your "time-series" data

3. **production_orders** - What's being manufactured
   * Links machines to the products they're making
   * Tracks order status and due dates

4. **maintenance_tickets** - When things go wrong
   * Preventive and corrective maintenance
   * Links back to machines via `machine_id`

### What you'll learn:

* How to generate realistic sample data with **pandas**
* How to work with **timestamps** and **date ranges**
* How **foreign keys** connect tables (machine_id appears in all 4 tables)
* How to write pandas DataFrames to **Delta tables** in Unity Catalog

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set random seed for reproducibility - same data every time you run this
np.random.seed(42)

# Fixed reference date so all timestamps are reproducible run-to-run
# (aligns with the production due_date anchor below — the dataset is set in mid-2026)
REFERENCE_DATE = datetime(2026, 7, 15)

# ─── 1. MACHINES TABLE ───
# 50 production machines across 3 production lines
# This is your "dimension table" - describes the equipment
machines = pd.DataFrame({
    'machine_id': range(1, 51),  # Primary key: unique ID for each machine
    'model': np.random.choice(['CNC-X1000', 'Lathe-Z500', 'Press-M200', 'Welder-A300'], 50),
    'line': np.random.choice(['Line-A', 'Line-B', 'Line-C'], 50),
    'install_date': pd.to_datetime('2020-01-01') + pd.to_timedelta(np.random.randint(0, 1000, 50), unit='D'),
    'location': np.random.choice(['Building-1', 'Building-2'], 50)
})

# ─── 2. SENSOR READINGS TABLE ───
# 10,000 IoT sensor readings over the last 30 days
# This is your "fact table" - time-series measurements
base_time = REFERENCE_DATE - timedelta(days=30)
sensor_readings = pd.DataFrame({
    'reading_id': range(1, 10001),  # Primary key
    'machine_id': np.random.randint(1, 51, 10000),  # Foreign key -> machines
    'ts': [base_time + timedelta(minutes=i*5) for i in range(10000)],  # One reading every 5 min
    'temperature_c': np.random.uniform(50, 90, 10000).round(1),  # Celsius
    'vibration_mm_s': np.random.uniform(0.5, 8.0, 10000).round(2),  # mm/s (high = bad)
    'load_pct': np.random.uniform(20, 95, 10000).round(1)  # Percentage of max capacity
})

# ─── 3. PRODUCTION ORDERS TABLE ───
# 200 manufacturing orders assigned to machines
# Links machines to what they're producing
production_orders = pd.DataFrame({
    'order_id': range(1001, 1201),  # Primary key: order number
    'machine_id': np.random.randint(1, 51, 200),  # Foreign key -> machines
    'product': np.random.choice(['Widget-A', 'Widget-B', 'Gear-X', 'Shaft-Y'], 200),
    'qty': np.random.randint(50, 500, 200),  # Quantity to manufacture
    'status': np.random.choice(['pending', 'in_progress', 'completed'], 200, p=[0.2, 0.3, 0.5]),
    'due_date': pd.to_datetime('2026-07-15') + pd.to_timedelta(np.random.randint(0, 60, 200), unit='D')
})

# ─── 4. MAINTENANCE TICKETS TABLE ───
# 120 maintenance tickets over the last 60 days
# Tracks both preventive maintenance and emergency repairs
ticket_base = REFERENCE_DATE - timedelta(days=60)
maintenance_tickets = pd.DataFrame({
    'ticket_id': range(1, 121),  # Primary key
    'machine_id': np.random.randint(1, 51, 120),  # Foreign key -> machines
    'opened_at': [ticket_base + timedelta(hours=i*12) for i in range(120)],  # One ticket every 12 hours
    'priority': np.random.choice(['low', 'medium', 'high', 'critical'], 120, p=[0.3, 0.4, 0.2, 0.1]),
    'status': np.random.choice(['open', 'in_progress', 'resolved'], 120, p=[0.3, 0.3, 0.4]),
    'description': np.random.choice([
        'Unusual vibration detected',
        'Temperature spike',
        'Scheduled maintenance',
        'Belt replacement needed',
        'Calibration required'
    ], 120)
})

# Summary of generated data
print("✅ Generated sample data:")
print(f"   • machines: {len(machines):,} rows")
print(f"   • sensor_readings: {len(sensor_readings):,} rows")
print(f"   • production_orders: {len(production_orders):,} rows")
print(f"   • maintenance_tickets: {len(maintenance_tickets):,} rows")
print(f"\n🔍 Preview the data below - notice how machine_id appears in all tables!")

# Preview each table (first 5 rows)
display(machines.head())
display(sensor_readings.head())
display(production_orders.head())
display(maintenance_tickets.head())

## ⚠️ Identifying High-Risk Machines

Before we load the data into Unity Catalog, let's do some **exploratory analysis**.

### What are we looking for?

Machines that show **both** high vibration AND high temperature are at risk of failure.

* **High vibration** (> 6.0 mm/s) - indicates mechanical issues
* **High temperature** (> 80°C) - indicates overheating

When both occur together, that's a strong signal for preventive maintenance.

### What you'll learn:

* How to **filter pandas DataFrames** with multiple conditions
* How to find **unique values** across filtered data
* Why **exploratory data analysis** (EDA) is important before loading data

In [0]:
# Filter for dangerous conditions: high vibration AND high temperature
# The & operator means BOTH conditions must be true
high_risk = sensor_readings[
    (sensor_readings['vibration_mm_s'] > 6.0) &   # Vibration threshold
    (sensor_readings['temperature_c'] > 80)       # Temperature threshold
]['machine_id'].unique()[:4].tolist()  # Get first 4 unique machine IDs

print(f"⚠️  High-risk machines (need immediate attention): {high_risk}")
print(f"   These {len(high_risk)} machines showed both high vibration and temperature")

# Turn that signal into work: give each high-risk machine an OPEN, HIGH-priority ticket, so it
# surfaces at the top of the app's alert queue in Lab 3 (sensor signal → ticket → app).
# Overwrite the first rows so the total stays 120.
for i, mid in enumerate(high_risk):
    maintenance_tickets.loc[i, ['machine_id', 'priority', 'status', 'description']] = [
        int(mid), 'high', 'open', 'High vibration + temperature — inspect before failure'
    ]
print(f"✅ Opened a high-priority ticket for each — they'll top the app's alert queue in Lab 3")

print(f"\n🔍 Try this: Change the thresholds above and re-run to find different machines!")

## 💾 Writing to Unity Catalog

Now we'll save our pandas DataFrames as **Delta tables** in Unity Catalog.

### What is Unity Catalog?

Unity Catalog is Databricks' **data governance layer**. It provides:
* 🛡️ **Centralized governance** - one place to manage all your data
* 🔐 **Access control** - who can read/write what
* 📄 **Lineage tracking** - where data comes from and goes to
* 🔍 **Search & discovery** - find tables across your organization

### What is a Delta table?

Delta Lake is an **open-source storage format** that adds:
* 📝 **ACID transactions** - reliable writes, no corruption
* ⏱️ **Time travel** - query historical versions
* 🚀 **Performance** - optimized for analytics
* 🔄 **Schema evolution** - columns can be added/modified

### What you'll learn:

* How to convert **pandas DataFrames** to **Spark DataFrames**
* How to write data as **Delta tables** with `.saveAsTable()`
* Why **mode('overwrite')** replaces existing data
* How Unity Catalog organizes data: `catalog.schema.table`

In [0]:
# Write all tables to Unity Catalog as Delta tables
# Loop through each DataFrame and save it
for name, df in [
    ('machines', machines),
    ('sensor_readings', sensor_readings),
    ('production_orders', production_orders),
    ('maintenance_tickets', maintenance_tickets)
]:
    # Convert pandas DataFrame to Spark DataFrame (required for Delta tables)
    sdf = spark.createDataFrame(df)
    
    # Write to Unity Catalog as a Delta table
    # mode('overwrite') replaces the table if it exists
    sdf.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{schema}.{name}")
    
    # Verify the write succeeded by counting rows
    count = spark.table(f"{CATALOG}.{schema}.{name}").count()
    print(f"✅ {name:25s} {count:,} rows")

print(f"\n✅ All four tables loaded into {CATALOG}.{schema}")
print(f"   Tables are now stored as Delta format with full ACID guarantees")

# Preview one of the tables to confirm it's readable
print(f"\n🔍 Sample from maintenance_tickets:")
pdf = spark.table(f"{CATALOG}.{schema}.maintenance_tickets").limit(5).toPandas()
display(pdf)

## ✅ 5. Verify Lab 1

Let's verify everything was created successfully and explore the Unity Catalog structure.

### What you'll see:

* The full Unity Catalog hierarchy (`catalog → schema → tables`)
* Row counts for each table
* The tables are ready for **Lab2** to sync to Postgres

### Things to try:

1. **Explore in the UI**: Click on [catalog_workshop](#catalog) to browse tables
2. **Query the data**: Try `SELECT * FROM catalog_workshop.schema_{your_name}.machines LIMIT 10`
3. **Check lineage**: In the Unity Catalog UI, click on a table to see its history
4. **Time travel**: Run `SELECT * FROM catalog_workshop.schema_{your_name}.machines VERSION AS OF 0`

In [0]:
# Verify the current user's schema and tables
print("="*70)
print(f"✅ LAB 1 COMPLETE - VERIFICATION")
print("="*70)

print(f"\n💾 Your data location: {CATALOG}.{schema}\n")

# Check only your schema's tables
expected_tables = ['machines', 'sensor_readings', 'production_orders', 'maintenance_tickets']
tables = spark.sql(f"SHOW TABLES IN {CATALOG}.{schema}").collect()
table_names = [t.tableName for t in tables]

all_ok = True
for tbl in expected_tables:
    if tbl in table_names:
        count = spark.table(f"{CATALOG}.{schema}.{tbl}").count()
        print(f"  ✅ {tbl:30s} {count:,} rows")
    else:
        print(f"  ❌ {tbl:30s} MISSING")
        all_ok = False

print(f"\n" + "="*70)
if all_ok:
    print(f"✅ SUCCESS! All 4 tables created and ready for Lab2.")
else:
    print(f"⚠️  Some tables are missing. Re-run cell 4 (data generation) and cell 8 (write to UC).")
print("="*70)
print(f"\n🚀 Next steps:")
print(f"   1. Open Lab2 to sync these tables to Lakebase Postgres")
print(f"   2. Enable Change Data Feed to track all changes")
print(f"   3. Build a real-time operational app on top of Postgres")
print(f"\n🔍 Explore: Click [catalog_workshop](#catalog) to browse your tables in the UI")